In [0]:
dbutils.widgets.text("p_data_source", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configs"

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

**Read the JSON file using spark dataframe reader API**

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
pitstops_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                     StructField("driverId", IntegerType(), True),
                                     StructField("stop", StringType(), True),
                                     StructField("lap", IntegerType(), True),
                                     StructField("time", IntegerType(), True),
                                     StructField("duration", IntegerType(), True),
                                     StructField("milliseconds", IntegerType(), True)])

In [0]:
%run "../Includes/comm_func" 

In [0]:
pitstops_df = spark.read.option("multiline", True).schema(pitstops_schema).json(f"{raw_folder_path}/{v_file_date}/pit_stops.json")

In [0]:
display(pitstops_df) #spark by default doesn't deal with multi-line files, so we added option("multiline", True)

raceId,driverId,stop,lap,time,duration,milliseconds
1053,839,1,1,null,null,30866
1053,20,1,3,null,null,32024
1053,854,1,5,null,null,51007
1053,853,1,12,null,null,31168
1053,842,1,14,null,null,31068
1053,20,2,20,null,null,31184
1053,854,2,21,null,null,32479
1053,20,3,22,null,null,39502
1053,853,2,23,null,null,31500
1053,852,1,25,null,null,30696


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
pitstops_final_df = pitstops_df.withColumnRenamed("raceId", "race_id").withColumnRenamed("driverId", "driver_id").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source)).withColumn("file_date", lit(v_file_date))

In [0]:
%sql
--DROP TABLE IF EXISTS f1_processed.pit_stops;

In [0]:
pitstops_final_df = pitstops_final_df.dropDuplicates(
    ["race_id", "driver_id", "stop"]
)

In [0]:
pitstops_final_df.write \
    .mode("append") \
    .format("delta") \
    .partitionBy("race_id") \
    .save(f"{processed_folder_path}/pitstops")

In [0]:
#pitstops_final_df.write.mode("overwrite").partitionBy("race_id").format("delta").saveAsTable("f1_processed.pit_stops")

In [0]:
#display(spark.read.parquet(f"{processed_folder_path}/pit_stops"))

In [0]:
merge_into_table(
    pitstops_final_df,
    "f1_processed",
    "pit_stops",
    "race_id",
    "target.race_id = source.race_id AND target.driver_id = source.driver_id AND target.stop = source.stop"
)

In [0]:
%sql
SELECT * FROM f1_processed.pit_stops
ORDER BY race_id DESC;

race_id,driver_id,stop,lap,time,duration,milliseconds,ingestion_date,data_source,file_date
1053,839,1,1,null,null,30866,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,20,1,3,null,null,32024,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,854,1,5,null,null,51007,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,853,1,12,null,null,31168,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,842,1,14,null,null,31068,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,20,2,20,null,null,31184,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,854,2,21,null,null,32479,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,20,3,22,null,null,39502,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,853,2,23,null,null,31500,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
1053,852,1,25,null,null,30696,2026-01-20T10:26:05.286317Z,Ergast API,2021-04-18
